# 01 — SEC Filings Downloader

**Job:** CIK → SEC submissions API → 13F-HR filings → *content-verified* information-table XML → `data/filings.parquet` + cached XML.

**Lessons baked into `src/sec.py` (from earlier failed attempts):**
1. **403 errors** — the SEC requires a descriptive `User-Agent` with contact info. Set yours in `src/config.py`.
2. **Never hardcode a `Host` header** — the old code sent `Host: data.sec.gov` to `www.sec.gov`, silently breaking Archives requests.
3. **`primary_doc.xml` is the cover page, not the holdings** — the old keyword search picked it first, which is why holdings came back empty. We now *download and verify* that a candidate XML actually contains `<infoTable>` rows before accepting it.
4. **Quarter = `reportDate`, not `filingDate`** — 13Fs are filed ~45 days after quarter end, so filing-date quarters are shifted by one.
5. **Amendments (13F-HR/A) supersede originals** — we keep the latest filing per report period.
6. **Rate limiting + retries** — SEC allows ≤10 req/s; we sleep between requests and retry 403/429/5xx with backoff instead of hiding errors in a bare `except`.

In [ ]:
# --- setup (run this cell first, every session) ---
import os, sys, pathlib

# Option A: Google Drive (data persists across sessions -- recommended)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: uploaded zip directly to Colab (data lost on disconnect)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# Option C: running locally
PROJECT_ROOT = pathlib.Path(__file__).resolve().parents[1] if "__file__" in dir() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# Install dependencies if needed (uncomment on Colab)
# import subprocess; subprocess.run(["pip", "install", "-q", "pyarrow", "requests", "pandas", "matplotlib", "numpy"])

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Project root:", PROJECT_ROOT)
print("Ready.")

In [ ]:
from src import config
from src.utils import make_session
from src.sec import build_filings_dataset

print("Manager:", config.MANAGER_NAME, "| CIK:", config.CIK)
print("User-Agent:", config.USER_AGENT)
assert "your_email" not in config.USER_AGENT, (
    "Edit USER_AGENT in src/config.py with your real name/email first — "
    "the SEC rejects anonymous requests with 403."
)

In [ ]:
session = make_session()
filings = build_filings_dataset(session, n_quarters=config.N_QUARTERS)
filings

In [ ]:
# Sanity checks: every quarter should have a verified holdings XML
assert filings["local_xml_path"].notna().all(), "Some filings have no verified infoTable XML"
assert filings["quarter"].is_unique
filings[["quarter", "form", "filingDate", "reportDate", "xml_file"]]

**Output:** `data/processed/filings.parquet` + one cached XML per quarter in `data/raw_xml/`. Next: `02_download_xml.ipynb`.